In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the Data
with open('/home/nfm/Desktop/rhome/nfm/ViT-Prisma/mynotebooks/org_clip_metrics_real/all_group_scores.json', 'r') as f:
    raw_data = json.load(f)

# 2. Convert to DataFrame
# Rows will be Heads (e.g., 'Layer 8 Head 0'), Columns will be Semantic Groups
rows = []
for head_id, head_info in raw_data.items():
    row = head_info['Scores'].copy()
    row['Head_Name'] = head_info['Head_Name']
    
    # Extract Layer and Head numbers for sorting
    parts = head_info['Head_Name'].split()
    row['Layer'] = int(parts[1])
    row['Head'] = int(parts[3])
    rows.append(row)

df = pd.DataFrame(rows)
df = df.sort_values(['Layer', 'Head']).reset_index(drop=True)

# Separate the scores from the metadata
score_cols = [c for c in df.columns if c not in ['Head_Name', 'Layer', 'Head']]

# 3. Identify Global Top N Groups (to prevent legend color explosion)
TOP_N = 10
global_sums = df[score_cols].sum().sort_values(ascending=False)
top_groups = global_sums.head(TOP_N).index.tolist()

# 4. Create a consistent global color palette
# Use a colorblind-friendly palette for the top groups, and grey for "Other"
colors = sns.color_palette("tab10", TOP_N)
color_map = {group: colors[i] for i, group in enumerate(top_groups)}
color_map['Other'] = '#d3d3d3' # Light grey for everything else

In [5]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- 1. Load the Data ---
with open('/home/nfm/Desktop/rhome/nfm/ViT-Prisma/mynotebooks/org_clip_metrics_real/all_group_scores.json', 'r') as f:
    raw_data = json.load(f)

rows = []
for head_id, head_info in raw_data.items():
    row = head_info['Scores'].copy()
    row['Head_Name'] = head_info['Head_Name']
    
    parts = head_info['Head_Name'].split()
    row['Layer'] = int(parts[1])
    row['Head'] = int(parts[3])
    rows.append(row)

df = pd.DataFrame(rows)
df = df.sort_values(['Layer', 'Head']).reset_index(drop=True)
score_cols = [c for c in df.columns if c not in ['Head_Name', 'Layer', 'Head']]

# --- 2. Dynamic Top Groups (The #1 group from every head) ---
top_groups_set = set()

for idx, row in df.iterrows():
    # Only look at the score columns
    scores = row[score_cols]
    if scores.max() > 0:  # Ensure the head actually has data
        top_1_group = scores.idxmax()
        top_groups_set.add(top_1_group)

# Convert to list and sort by global overall dominance to make the legend tidy
global_sums = df[score_cols].sum()
top_groups = sorted(list(top_groups_set), key=lambda g: global_sums[g], reverse=True)

print(f"Total unique top-1 groups found across 48 heads: {len(top_groups)}")

# --- 3. Dynamic Color Palette ---
# If we have more than 20 groups, tab20 won't be enough, so we use 'husl' which generates N distinct colors
if len(top_groups) <= 10:
    colors = sns.color_palette("tab10", len(top_groups))
elif len(top_groups) <= 20:
    colors = sns.color_palette("tab20", len(top_groups))
else:
    colors = sns.color_palette("husl", len(top_groups))

color_map = {group: colors[i] for i, group in enumerate(top_groups)}
color_map['Other'] = '#d3d3d3' # Light grey

# --- 4. Plotting Design A (Stacked Bars) ---
def plot_design_a(df, top_groups, color_map, save_path):
    fig, axes = plt.subplots(4, 12, figsize=(24, 8), sharex=True, sharey=True)
    fig.subplots_adjust(wspace=0.1, hspace=0.4, bottom=0.25) # Increased bottom margin for dynamic legend
    
    layers = sorted(df['Layer'].unique())
    
    for i, layer in enumerate(layers):
        layer_df = df[df['Layer'] == layer]
        for j in range(12):
            ax = axes[i, j]
            if j < len(layer_df):
                row = layer_df.iloc[j]
                
                scores = {g: row[g] for g in top_groups}
                scores['Other'] = sum(row[g] for g in score_cols if g not in top_groups)
                
                total = sum(scores.values())
                if total > 0:
                    scores = {k: (v / total) * 100 for k, v in scores.items()}
                
                left = 0
                for group, score in scores.items():
                    if score > 0:
                        ax.barh(0, score, left=left, color=color_map[group], edgecolor='none')
                        left += score
                        
            ax.set_yticks([])
            ax.set_xticks([])
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_visible(False)
            ax.spines['bottom'].set_visible(False)
            
            if i == 0:
                ax.set_title(f'H{j}', fontsize=10)
        
        axes[i, 0].set_ylabel(f'L{layer}', rotation=0, labelpad=20, va='center', fontsize=12, fontweight='bold')

    handles = [plt.Rectangle((0,0),1,1, color=color_map[g]) for g in top_groups + ['Other']]
    
    # Calculate rows needed for legend dynamically to prevent overlapping
    ncols = 6
    fig.legend(handles, top_groups + ['Other'], loc='lower center', ncol=ncols, bbox_to_anchor=(0.5, 0.02), frameon=False, fontsize=10)
    
    plt.suptitle("Semantic Group Distribution per Attention Head", fontsize=16, y=0.95)
    
    # --- FIX FOR FILE NOT FOUND ERROR ---
    # This automatically creates the 'plots/head_fun/' directory if it doesn't exist
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Successfully saved plot to {save_path}")
    plt.close()

# Define your save path
my_save_path = '/home/nfm/Desktop/rhome/nfm/ViT-Prisma/mynotebooks/org_clip_metrics_real/plots/head_fun/design_a_stacked_bars.pdf'

# Execute the plot
plot_design_a(df, top_groups, color_map, my_save_path)

Total unique top-1 groups found across 48 heads: 30
Successfully saved plot to /home/nfm/Desktop/rhome/nfm/ViT-Prisma/mynotebooks/org_clip_metrics_real/plots/head_fun/design_a_stacked_bars.pdf


In [6]:
def plot_design_b(df):
    # For a heatmap, we might want to show more groups, e.g., Top 20
    top_20_groups = df[score_cols].sum().sort_values(ascending=False).head(20).index.tolist()
    
    # Prepare matrix
    heatmap_data = df.set_index('Head_Name')[top_20_groups]
    
    # Optional: Row normalize the data if you want to see relative dominance per head rather than absolute magnitude
    heatmap_data = heatmap_data.div(heatmap_data.sum(axis=1), axis=0).fillna(0)
    
    plt.figure(figsize=(14, 10))
    
    # Use a perceptually uniform colormap like 'viridis' or 'Blues' for papers
    ax = sns.heatmap(heatmap_data, cmap='Blues', cbar_kws={'label': 'Normalized Attention Score'},
                     linewidths=0.5, linecolor='white')
    
    # Add horizontal lines to separate layers visually (every 12 heads)
    for i in range(12, 48, 12):
        ax.axhline(i, color='black', lw=1.5)
        
    plt.title("Top 20 Semantic Groups Across All Heads", fontsize=14, pad=20)
    plt.ylabel("Attention Head", fontsize=12)
    plt.xlabel("Semantic Group", fontsize=12)
    
    # Formatting ticks
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(fontsize=8)
    
    plt.tight_layout()
    plt.savefig('/home/nfm/Desktop/rhome/nfm/ViT-Prisma/mynotebooks/org_clip_metrics_real/plots/head_fun/design_b_heatmap.png', dpi=300, bbox_inches='tight')
    plt.close()

plot_design_b(df)

In [7]:
def plot_design_c(df, top_groups, color_map):
    fig, axes = plt.subplots(4, 12, figsize=(22, 8), sharex=True, sharey=True)
    fig.subplots_adjust(wspace=0.1, hspace=0.4, bottom=0.2)
    
    layers = sorted(df['Layer'].unique())
    max_score = df[score_cols].max().max() # Used to scale bubbles
    
    for i, layer in enumerate(layers):
        layer_df = df[df['Layer'] == layer]
        for j in range(12):
            ax = axes[i, j]
            if j < len(layer_df):
                row = layer_df.iloc[j]
                
                # Get top 3 groups for THIS SPECIFIC HEAD
                head_top_3 = row[score_cols].sort_values(ascending=False).head(3)
                
                # Plot them as bubbles
                for rank, (group, score) in enumerate(head_top_3.items()):
                    if score > 0:
                        # Map to global color if in top_groups, else Other
                        c = color_map.get(group, color_map['Other'])
                        # Scale bubble size (multiply by a constant for visibility)
                        s = (score / max_score) * 600 
                        ax.scatter(rank, 0, s=s, color=c, alpha=0.8, edgecolors='white')
                        
            # Clean axes completely
            ax.set_xlim(-0.5, 2.5)
            ax.set_ylim(-0.1, 0.1)
            ax.axis('off')
            
            if i == 0:
                ax.set_title(f'H{j}', fontsize=10)
                
        # Layer labels
        axes[i, 0].text(-1.5, 0, f'L{layer}', va='center', fontsize=12, fontweight='bold')

    # Global legend (using dummy scatter points)
    handles = [plt.scatter([], [], s=100, color=color_map[g]) for g in top_groups + ['Other']]
    fig.legend(handles, top_groups + ['Other'], loc='lower center', ncol=6, bbox_to_anchor=(0.5, 0.05), frameon=False, scatterpoints=1)
    
    plt.suptitle("Top 3 Semantic Concepts per Head (Size = Magnitude)", fontsize=16, y=0.95)
    plt.savefig('/home/nfm/Desktop/rhome/nfm/ViT-Prisma/mynotebooks/org_clip_metrics_real/plots/head_fun/design_c_bubbles.png', dpi=300, bbox_inches='tight')
    plt.close()

plot_design_c(df, top_groups, color_map)

In [18]:
import json

DLA_PATH = "/home/nfm/Desktop/rhome/nfm/ViT-Prisma/mynotebooks/org_clip_metrics_real/head_top_texts.json"
ATTR_PATH = "/home/nfm/Desktop/rhome/nfm/ViT-Prisma/mynotebooks/head_top_texts_attr_patch.json"

HEAD_IDXS = [0, 1, 2, 3]   # even number of heads; paired two per table block
TOP_N = 10
COL_WIDTH = "3.2cm"        # width of each text column
SHOW_COUNT = False          # append "(list length)" after each text


def _escape_latex(s):
    repl = {
        "\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$",
        "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}",
        "~": r"\textasciitilde{}", "^": r"\textasciicircum{}",
    }
    return "".join(repl.get(c, c) for c in str(s))


def _load():
    with open(DLA_PATH) as f:
        dla = json.load(f)
    with open(ATTR_PATH) as f:
        attr = json.load(f)
    return dla, attr


def _top(d, head_idx, top_n):
    """Top-N texts for a head, selected by length of the value list (desc)."""
    key = f"Head_{head_idx}"
    if key not in d:
        raise KeyError(f"{key} not found")
    items = d[key]["Top_Texts"].items()
    ranked = sorted(
        items,
        key=lambda kv: (len(kv[1]), kv[1][0] if kv[1] else float("-inf")),
        reverse=True,
    )[:top_n]
    return [(t, len(v)) for t, v in ranked]


def _cell(entry):
    if entry is None:
        return ""
    text, cnt = entry
    out = _escape_latex(text)
    return f"{out}~({cnt})" if SHOW_COUNT else out


def heads_comparison_table(head_idxs, top_n=10):
    """LaTeX (table only). Heads are paired two per block; each table row compares
    the same rank across the two heads, DLA vs attribution patching."""
    dla, attr = _load()
    blocks = []

    for a, b in zip(head_idxs[0::2], head_idxs[1::2]):
        heads = [a, b]
        names = [dla[f"Head_{h}"].get("Head_Name", f"Head_{h}") for h in heads]
        tops = {h: (_top(dla, h, top_n), _top(attr, h, top_n)) for h in heads}
        n = max(len(t) for pair in tops.values() for t in pair)

        lines = [
            r"\begin{table}[ht]",
            r"\centering",
            r"\scriptsize",
            rf"\begin{{tabular}}{{c p{{{COL_WIDTH}}} p{{{COL_WIDTH}}} | p{{{COL_WIDTH}}} p{{{COL_WIDTH}}}}}",
            r"\toprule",
            rf" & \multicolumn{{2}}{{c|}}{{\textbf{{{_escape_latex(names[0])}}}}} "
            rf"& \multicolumn{{2}}{{c}}{{\textbf{{{_escape_latex(names[1])}}}}} \\",
            r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}",
            r"\textbf{Rank} & \textbf{DLA} & \textbf{Attr.\ Patch} & \textbf{DLA} & \textbf{Attr.\ Patch} \\",
            r"\midrule",
        ]
        for i in range(n):
            cells = []
            for h in heads:
                dla_top, attr_top = tops[h]
                cells.append(_cell(dla_top[i] if i < len(dla_top) else None))
                cells.append(_cell(attr_top[i] if i < len(attr_top) else None))
            lines.append(f"{i + 1} & " + " & ".join(cells) + r" \\")
        lines += [
            r"\bottomrule",
            r"\end{tabular}",
            rf"\caption{{Top {top_n} texts (by value-list length) for "
            rf"{_escape_latex(names[0])} and {_escape_latex(names[1])} "
            r"(DLA vs.\ attribution patching; counts in parentheses).}}",
            rf"\label{{tab:heads_{a}_{b}_top_texts}}",
            r"\end{table}",
        ]
        blocks.append("\n".join(lines))

    return "\n\n".join(blocks)


# print(heads_comparison_table(HEAD_IDXS, TOP_N))


HEAD_IDXS = [46, 35]
TOP_N = 5
print(heads_comparison_table(HEAD_IDXS, TOP_N))

\begin{table}[ht]
\centering
\scriptsize
\begin{tabular}{c p{3.2cm} p{3.2cm} | p{3.2cm} p{3.2cm}}
\toprule
 & \multicolumn{2}{c|}{\textbf{Layer 11 Head 10}} & \multicolumn{2}{c}{\textbf{Layer 10 Head 11}} \\
\cmidrule(lr){2-3}\cmidrule(lr){4-5}
\textbf{Rank} & \textbf{DLA} & \textbf{Attr.\ Patch} & \textbf{DLA} & \textbf{Attr.\ Patch} \\
\midrule
1 & An image with cold green tones & An image with cold green tones & Bustling city waterfront & Image with a five people \\
2 & Image with a green color & Image with a blue color & Busy market square & Image with a seven people \\
3 & Photograph with a green color palette & Image with a green color & Bustling city square & Image with a six people \\
4 & Image with a yellow color & Image with a purple color & Bustling city from above & Image with a four people \\
5 & Image with a pink color & Image with a yellow color & Crowded and bustling scene & Image with three people \\
\bottomrule
\end{tabular}
\caption{Top 5 texts (by value-list length)

In [19]:
import json
import re

DLA_PATH = "/home/nfm/Desktop/rhome/nfm/ViT-Prisma/mynotebooks/org_clip_metrics_real/head_top_texts.json"
ATTR_PATH = "/home/nfm/Desktop/rhome/nfm/ViT-Prisma/mynotebooks/head_top_texts_attr_patch.json"
SAE_PATH = "/home/nfm/Desktop/rhome/nfm/ViT-Prisma/mynotebooks/head_top_texts_sae.json"

HEAD_IDXS = [0, 1, 2, 3]   # even number of heads; paired two per table block
TOP_N = 10
COL_WIDTH = "2.6cm"        # width of each text column (6 text columns per block)


def _escape_latex(s):
    repl = {
        "\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$",
        "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}",
        "~": r"\textasciitilde{}", "^": r"\textasciicircum{}",
    }
    return "".join(repl.get(c, c) for c in str(s))


def _load():
    with open(DLA_PATH) as f:
        dla = json.load(f)
    with open(ATTR_PATH) as f:
        attr = json.load(f)
    with open(SAE_PATH) as f:
        sae = json.load(f)
    return dla, attr, sae


def _dedup_take_n(ordered_pairs, n):
    """ordered_pairs: list of (text, score) already sorted. Keep the first text of
    each distinct-score group (drop exact-duplicate scores), take the first n."""
    out, last = [], object()  # sentinel that won't equal any score
    for text, score in ordered_pairs:
        if score != last:
            out.append(text)
            last = score
            if len(out) == n:
                break
    return out


def _sae_key(dla, head_idx):
    """Map head index -> SAE key like 'layer8_head0' using the DLA Head_Name."""
    name = dla[f"Head_{head_idx}"].get("Head_Name", "")
    m = re.search(r"Layer\s+(\d+)\s+Head\s+(\d+)", name)
    if m:
        return f"layer{m.group(1)}_head{m.group(2)}"
    return f"layer{8 + head_idx // 12}_head{head_idx % 12}"  # fallback


def _top_for_head(dla, attr, sae, head_idx, n):
    key = f"Head_{head_idx}"
    name = dla[key].get("Head_Name", key)

    # DLA / Attr: value is a list -> rank by list length (desc), dedup on length
    def list_pairs(d):
        items = [(t, len(v)) for t, v in d[key]["Top_Texts"].items()]
        items.sort(key=lambda kv: kv[1], reverse=True)
        return items

    dla_texts = _dedup_take_n(list_pairs(dla), n)
    attr_texts = _dedup_take_n(list_pairs(attr), n)

    # SAE: value is a float -> already sorted desc, dedup on the float score
    sk = _sae_key(dla, head_idx)
    sae_pairs = sorted(sae[sk].items(), key=lambda kv: kv[1], reverse=True)
    sae_texts = _dedup_take_n(sae_pairs, n)

    return name, dla_texts, attr_texts, sae_texts


def heads_comparison_table(head_idxs, top_n=10):
    """LaTeX (table only). Heads paired two per block; each row compares the same
    rank across two heads, with DLA / Attribution Patching / SAE columns each."""
    dla, attr, sae = _load()
    w = COL_WIDTH
    blocks = []

    for a, b in zip(head_idxs[0::2], head_idxs[1::2]):
        cols = {h: _top_for_head(dla, attr, sae, h, top_n) for h in (a, b)}
        n = max(len(x) for _, *lists in cols.values() for x in lists) if cols else 0
        name_a, name_b = cols[a][0], cols[b][0]

        lines = [
            r"\begin{table}[ht]",
            r"\centering",
            r"\scriptsize",
            rf"\begin{{tabular}}{{c p{{{w}}} p{{{w}}} p{{{w}}} | p{{{w}}} p{{{w}}} p{{{w}}}}}",
            r"\toprule",
            rf" & \multicolumn{{3}}{{c|}}{{\textbf{{{_escape_latex(name_a)}}}}} "
            rf"& \multicolumn{{3}}{{c}}{{\textbf{{{_escape_latex(name_b)}}}}} \\",
            r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}",
            r"\textbf{Rank} & \textbf{DLA} & \textbf{Attr.\ Patch} & \textbf{SAE}"
            r" & \textbf{DLA} & \textbf{Attr.\ Patch} & \textbf{SAE} \\",
            r"\midrule",
        ]
        for i in range(n):
            cells = []
            for h in (a, b):
                _, dla_t, attr_t, sae_t = cols[h]
                for lst in (dla_t, attr_t, sae_t):
                    cells.append(_escape_latex(lst[i]) if i < len(lst) else "")
            lines.append(f"{i + 1} & " + " & ".join(cells) + r" \\")
        lines += [
            r"\bottomrule",
            r"\end{tabular}",
            rf"\caption{{Top {top_n} texts for {_escape_latex(name_a)} and "
            rf"{_escape_latex(name_b)} (DLA vs.\ attribution patching vs.\ SAE; "
            r"exact-duplicate scores collapsed).}}",
            rf"\label{{tab:heads_{a}_{b}_top_texts}}",
            r"\end{table}",
        ]
        blocks.append("\n".join(lines))

    return "\n\n".join(blocks)

HEAD_IDXS = [46, 35]
TOP_N = 5
print(heads_comparison_table(HEAD_IDXS, TOP_N))

\begin{table}[ht]
\centering
\scriptsize
\begin{tabular}{c p{2.6cm} p{2.6cm} p{2.6cm} | p{2.6cm} p{2.6cm} p{2.6cm}}
\toprule
 & \multicolumn{3}{c|}{\textbf{Layer 11 Head 10}} & \multicolumn{3}{c}{\textbf{Layer 10 Head 11}} \\
\cmidrule(lr){2-4}\cmidrule(lr){5-7}
\textbf{Rank} & \textbf{DLA} & \textbf{Attr.\ Patch} & \textbf{SAE} & \textbf{DLA} & \textbf{Attr.\ Patch} & \textbf{SAE} \\
\midrule
1 & An image with cold green tones & An image with cold green tones & Image with a black color & Bustling city waterfront & Image with a five people & Image of an airplane \\
2 & Image with a green color & Image with a blue color & A platinum silver color & Busy market square & Image with a seven people & Image of a motorcycle \\
3 & Photograph with a green color palette & Image with a green color & Image with a white color & Bustling city from above & Image with a six people & Picture taken in Vietnam \\
4 & Image with a yellow color & Image with a purple color & Image with a yellow color & Crow

In [ ]:
import json
import re

# ---- config ----
INPUT_JSON = "/home/nfm/Desktop/rhome/nfm/ViT-Prisma/mynotebooks/org_clip_metrics_real/all_group_scores.json"   # change to your file path
TOP_N = 5
MAX_NAME = 20    

In [24]:
import json
import re

# ---- config ----
INPUT_JSON = "/home/nfm/Desktop/rhome/nfm/ViT-Prisma/mynotebooks/org_clip_metrics_real/all_group_scores.json"   # change to your file path
TOP_N = 2
MAX_NAME = 22                # truncate long group names so they don't wrap much
# ----------------

def esc(s):
    repl = {'&': r'\&', '%': r'\%', '$': r'\$', '#': r'\#',
            '_': r'\_', '{': r'\{', '}': r'\}', '~': r'\textasciitilde{}',
            '^': r'\textasciicircum{}'}
    return ''.join(repl.get(c, c) for c in s)

def shorten(name):
    name = name.strip()
    if len(name) > MAX_NAME:
        name = name[:MAX_NAME - 1].rstrip() + "\u2026"
    return name

with open(INPUT_JSON) as f:
    data = json.load(f)

grid = {}
layers, heads = set(), set()
for entry in data.values():
    m = re.search(r"Layer\s+(\d+)\s+Head\s+(\d+)", entry["Head_Name"])
    layer, head = int(m.group(1)), int(m.group(2))
    layers.add(layer); heads.add(head)
    top = sorted(entry["Scores"].items(), key=lambda kv: kv[1], reverse=True)[:TOP_N]
    grid.setdefault(layer, {})[head] = top

layers = sorted(layers); heads = sorted(heads)

def cell(top):
    if not top:
        return "--"
    return r" \newline ".join(esc(shorten(g)) for g, _ in top)

# narrow head column + auto-width X columns that fill exactly \textwidth
col_spec = "|c|" + "*{%d}{>{\\raggedright\\arraybackslash}X|}" % len(layers)

lines = []
lines.append(r"\begin{table}[ht]")
lines.append(r"\centering")
lines.append(r"\renewcommand{\arraystretch}{1.1}")
lines.append(r"\setlength{\tabcolsep}{3pt}")
lines.append(r"\scriptsize")
lines.append(rf"\begin{{tabularx}}{{\textwidth}}{{{col_spec}}}")
lines.append(r"\hline")
lines.append(r"\textbf{Head} & " +
             " & ".join(rf"\textbf{{Layer {l}}}" for l in layers) + r" \\")
lines.append(r"\hline")
for h in heads:
    row = [rf"\textbf{{{h}}}"] + [cell(grid.get(l, {}).get(h, [])) for l in layers]
    lines.append(" & ".join(row) + r" \\")
    lines.append(r"\hline")
lines.append(r"\end{tabularx}")
lines.append(r"\caption{Top 2 semantic groups per attention head across layers.}")
lines.append(r"\label{tab:head_semantics}")
lines.append(r"\end{table}")

latex = "\n".join(lines)
with open("table.tex", "w") as f:
    f.write(latex)
print(latex)

\begin{table}[ht]
\centering
\renewcommand{\arraystretch}{1.1}
\setlength{\tabcolsep}{3pt}
\scriptsize
\begin{tabularx}{\textwidth}{|c|*{4}{>{\raggedright\arraybackslash}X|}}
\hline
\textbf{Head} & \textbf{Layer 8} & \textbf{Layer 9} & \textbf{Layer 10} & \textbf{Layer 11} \\
\hline
\textbf{0} & Locations \& Places \newline Letters, Text \& Writi… & Timekeeping \& Clocks \newline Futuristic \& Science… & Vintage \& Retro Aesth… \newline Locations \& Places & Locations \& Places \newline Natural Landscapes \\
\hline
\textbf{1} & Facial Expressions \&… \newline Locations \& Places & Vintage \& Retro Aesth… \newline Artistic Styles \& Mov… & Locations \& Places \newline Futuristic \& Science… & Reflections \& Mirrors \newline Mood \& Atmosphere (Se… \\
\hline
\textbf{2} & Sky \& Space \newline Stars \& Star-shaped & Patterns, Textures \&… \newline Artistic Styles \& Mov… & Plants \& Botanical \newline Headwear \& Headgear & Fantasy, Magic \& Surr… \newline Mood \& Atmosphere (Dr… \\
\hlin

In [8]:
import json
import random

PATH = "/home/nfm/Desktop/rhome/nfm/ViT-Prisma/mynotebooks/groups_output_combined_new.json"
TOP_N = 5        # texts shown per group
SEED = 42        # set to None for a different random pick each run

with open(PATH) as f:
    data = json.load(f)

groups = data["groups"]

# only groups with at least TOP_N texts (ignore imagenet_classes entirely)
eligible = [name for name, g in groups.items() if len(g.get("texts", [])) >= TOP_N]
assert len(eligible) >= 4, "Not enough groups with >= %d texts" % TOP_N

rng = random.Random(SEED)
chosen = rng.sample(eligible, 4)


def tex_escape(s):
    s = s.replace("\\", r"\textbackslash{}")
    for a, b in {
        "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#",
        "_": r"\_", "{": r"\{", "}": r"\}",
        "~": r"\textasciitilde{}", "^": r"\textasciicircum{}",
    }.items():
        s = s.replace(a, b)
    return s


def cell(name):
    # plain p{} cell: bold header, then one text per line via \newline (no makecell)
    texts = groups[name]["texts"][:TOP_N]
    lines = [r"\textbf{%s}" % tex_escape(name)] + [tex_escape(t) for t in texts]
    return r"\newline ".join(lines)


rows = [
    "%s & %s \\\\ \\hline" % (cell(chosen[0]), cell(chosen[1])),
    "%s & %s \\\\ \\hline" % (cell(chosen[2]), cell(chosen[3])),
]

latex = r"""\begin{table}[htpb!]
\centering
\footnotesize
\setlength{\tabcolsep}{3pt}
\renewcommand{\arraystretch}{1.05}
\begin{tabular}{|p{0.34\textwidth}|p{0.34\textwidth}|}
\hline
%s
%s
\end{tabular}
\caption{Top %d texts for four randomly selected groups.}
\label{tab:random_groups}
\end{table}
""" % (rows[0], rows[1], TOP_N)

print(latex)

\begin{table}[htpb!]
\centering
\footnotesize
\setlength{\tabcolsep}{3pt}
\renewcommand{\arraystretch}{1.05}
\begin{tabular}{|p{0.34\textwidth}|p{0.34\textwidth}|}
\hline
\textbf{Graffiti \& Street Art}\newline A graffiti\newline A graffiti with a sentence\newline A mural\newline Artwork featuring graffiti-like designs\newline Bold graffiti & \textbf{Sky \& Space}\newline A constellation\newline A crescent moon\newline A galaxy\newline A meteor\newline A quasar \\ \hline
\textbf{Mood \& Atmosphere (Dramatic / Mysterious)}\newline A dramatic image\newline A volcano\newline Atmospheric haze\newline Atmospheric mood\newline Cinematic portrait with dramatic lighting & \textbf{Spirals, Swirls \& Vortexes}\newline A coil\newline A helix\newline A shell (of a snail or a nut)\newline A snail\newline A spiral \\ \hline
\end{tabular}
\caption{Top 5 texts for four randomly selected groups.}
\label{tab:random_groups}
\end{table}

